In [ ]:
# Install dependencies for this notebook

try:
    import awkward
    import hist
    import uproot
    import mplhep
except ModuleNotFoundError:
    import sys

    if sys.platform.startswith("emscripten"):
        import micropip

        await micropip.install("awkward==2.8.5")
        await micropip.install("hist")
        await micropip.install("uproot")
        await micropip.install("mplhep==0.4")
    else:
        %pip install awkward hist uproot mplhep

# Histogram libraries

Mainstream Python has libraries for filling histograms.

## NumPy

NumPy, for instance, has an `np.histogram` function.

In [ ]:
import uproot
import numpy as np

tree = uproot.open("data/uproot-Zmumu.root")["events"]

np.histogram(tree["M"].array())

Because of NumPy’s prominence, this 2-tuple of arrays (bin contents and edges) is a widely recognized histogram format, though it lacks many of the features high-energy physicists expect (under/overflow, axis labels, uncertainties, etc.).

## Matplotlib

Matplotlib also has a `plt.hist` function.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(tree["M"].array())

In addition to the same bin contents and edges as NumPy, Matplotlib includes a plottable graphic.

## Boost-histogram and hist

The main feature that these functions lack (without some effort) is refillability. High-energy physicists usually want to fill histograms with more data than can fit in memory, which means setting bin intervals on an empty container and filling it in batches (sequentially or in parallel).

Boost-histogram is a library designed for that purpose. It is intended as an infrastructure component. You can explore its “low-level” functionality upon importing it:

A more user-friendly layer (with plotting, for instance) is provided by a library called `hist`.

In [ ]:
import hist

h = hist.Hist(hist.axis.Regular(120, 60, 120, name="mass"))

h.fill(tree["M"].array())

h.plot()

## Universal Histogram Indexing (UHI)

There is an attempt within Scikit-HEP to standardize what array-like slices mean for a histogram. ([See documentation](https://uhi.readthedocs.io/en/latest/indexing.html))

Naturally, integer slices should select a range of bins,

In [ ]:
h[10:110].plot()

but often you want to select bins by coordinate value

In [ ]:
# Explicit version
# h[hist.loc(90) :].plot()

# Short version
h[90j:].plot()

or rebin by a factor,

In [ ]:
# Explicit version
# h[:: hist.rebin(2)].plot()

# Short version
h[::2j].plot()

or sum over a range.

In [ ]:
# Explicit version
# h[hist.loc(80) : hist.loc(100) : sum]

# Short version
h[90j:100j:sum]

Things get more interesting when a histogram has multiple dimensions.

In [ ]:
import hist
import numpy as np

rng = np.random.default_rng(seed=42)
x_data = rng.normal(loc=0.0, scale=1.0, size=10_000)
y_data = rng.normal(loc=0.5, scale=1.5, size=10_000)

h = hist.Hist(
    hist.axis.Regular(bins=50, start=-4, stop=4, name="X", label="X axis"),
    hist.axis.Regular(bins=50, start=-4, stop=6, name="Y", label="Y axis"),
)

h.fill(X=x_data, Y=y_data)

h.plot2d();

A histogram object can have more dimensions than you can reasonably visualize—you can slice, rebin, and project it into something visual later.

# Exercise

Let's go back to the big file we were using above. We restricted to events with exactly two muons and computed the dimuon masses. Now, we will use Awkward array to consider all possible combinations of two muons in each event. We will compute the masses and plot them. We have some code that does this below.

Make the following modifications to the code below.
1. Restrict to only muons of opposite charge.
2. Zoom in on the Z-peak and do some rebinning until you like how it looks.
3. If you have time, try to pick only the pair of muons with opposite charge with a mass closest to the Z boson (91 GeV). Hint: You'll need `ak.argmin` with `keepdims=True`.

:::{note} Answer (Don't look until you're done)
:class: dropdown

- For 1, you can use `cut = (mu1.charge != mu2.charge)`.
- For 2, it's up to you, but it would be something like `h[80j:100j:2j].plot()`.
- For 3, you can use `which = ak.argmin(abs(mass - zmass), axis=1, keepdims=True)`.
:::

In [ ]:
# Modify this code

import uproot
import awkward as ak
import numpy as np

file = uproot.open("data/Run2012BC_DoubleMuParked_Muons_1M_events.root")
tree = file["Events"]

arrays = tree.arrays(filter_name="/Muon_(pt|eta|phi|charge)/", entry_stop=1_000)

muons = ak.zip(
    {
        "pt": arrays["Muon_pt"],
        "eta": arrays["Muon_eta"],
        "phi": arrays["Muon_phi"],
        "charge": arrays["Muon_charge"],
    }
)

# Construct all possible pairs
pairs = ak.combinations(muons, 2)

# Separate them into first and second muon
mu1, mu2 = ak.unzip(pairs)

mass = np.sqrt(
    2 * mu1.pt * mu2.pt * (np.cosh(mu1.eta - mu2.eta) - np.cos(mu1.phi - mu2.phi))
)

# mass is a jagged array, so we need to flatten/ravel it in order to construct the histogram
h = hist.Hist(hist.axis.Regular(120, 0, 120, label="mass [GeV]"))
h.fill(ak.ravel(mass))
h.plot()